<a href="https://colab.research.google.com/github/aaronc09/pediatric-lupus-flare-prediction-gene-expression-ml/blob/main/notebooks/02_run_lupus_ml_pipeline_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
# =========================================================
# Pediatric Lupus Flare Prediction (GSE65391)
# Author: Aaron Choi
#
# Colab runner for nested cross-validation ML pipeline
# Main pipeline script:
# scripts/lupus_nested_cv_pipeline.py
# =========================================================

# ====================================
# 1. MOUNT GOOGLE DRIVE
# ====================================
from google.colab import drive
drive.mount("/content/drive")

# ====================================
# 2. INSTALL DEPENDENCIES
# ====================================
!pip uninstall -y umap-learn hdbscan > /dev/null 2>&1

!pip install -q \
  numpy==2.0.2 pandas==2.2.2 matplotlib==3.9.2 seaborn==0.13.2 \
  scikit-learn==1.5.2 xgboost==2.1.1 shap==0.46.0 optuna==4.0.0 \
  requests==2.32.4 pyarrow==17.0.0 openpyxl==3.1.5

# ====================================
# 3. CLONE OR UPDATE GITHUB REPO
# ====================================
import os
from pathlib import Path

REPO_PATH = "/content/pediatric-lupus-flare-prediction-gene-expression-ml"
REPO_URL = "https://github.com/aaronc09/pediatric-lupus-flare-prediction-gene-expression-ml.git"

if not os.path.exists(REPO_PATH):
    print("Cloning GitHub repo...")
    !git clone "$REPO_URL"
else:
    print("Repo already exists. Pulling latest changes...")
    %cd $REPO_PATH
    !git pull

# ====================================
# 4. VERIFY REPO CONTENTS
# ====================================
print("\nRepo contents:")
!ls "$REPO_PATH"

print("\nScripts folder contents:")
!ls "$REPO_PATH/scripts"

# ====================================
# 5. IMPORTS
# ====================================
import sys

# ====================================
# 6. CONFIGURE DETERMINISTIC EXECUTION
# ====================================
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

# ====================================
# 7. PATHS
# ====================================
SCRIPT_PATH = f"{REPO_PATH}/scripts/lupus_nested_cv_pipeline.py"
TEMP_SCRIPT_PATH = f"{REPO_PATH}/scripts/lupus_nested_cv_pipeline_colab_run.py"
DATA_PATH   = "/content/drive/MyDrive/lupus_project_outputs/lupus_final_df.pkl"
OUTPUT_DIR  = "/content/drive/MyDrive/lupus_ml_outputs"
FIGURES_DIR = os.path.join(OUTPUT_DIR, "figures")

# ====================================
# 8. VALIDATE PATHS
# ====================================
if not os.path.exists(REPO_PATH):
    raise FileNotFoundError(f"Repo path does not exist: {REPO_PATH}")

if not os.path.exists(SCRIPT_PATH):
    raise FileNotFoundError(f"Script file does not exist: {SCRIPT_PATH}")

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f"Processed dataset does not exist: {DATA_PATH}")

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
Path(FIGURES_DIR).mkdir(parents=True, exist_ok=True)

# ====================================
# 9. DELETE OLD PNG FIGURES
# ====================================
deleted_pngs = 0
for fname in os.listdir(FIGURES_DIR):
    if fname.lower().endswith(".png"):
        fpath = os.path.join(FIGURES_DIR, fname)
        try:
            os.remove(fpath)
            deleted_pngs += 1
        except OSError as e:
            print(f"Could not delete {fpath}: {e}")

print(f"\nDeleted {deleted_pngs} old PNG file(s) from: {FIGURES_DIR}")

# ====================================
# 10. MOVE INTO REPO FOLDER
# ====================================
%cd $REPO_PATH

# ====================================
# 11. CONFIGURE ENVIRONMENT VARIABLES
# ====================================
os.environ["LUPUS_DATA_PATH"] = DATA_PATH
os.environ["LUPUS_OUTPUT_DIR"] = OUTPUT_DIR

print("\nConfiguration:")
print("REPO_PATH:", REPO_PATH)
print("SCRIPT_PATH:", SCRIPT_PATH)
print("TEMP_SCRIPT_PATH:", TEMP_SCRIPT_PATH)
print("DATA_PATH:", DATA_PATH)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("FIGURES_DIR:", FIGURES_DIR)

# ====================================
# 12. CREATE TEMP RUN SCRIPT WITH FIGURES ENABLED
#    (GitHub file remains unchanged)
# ====================================
with open(SCRIPT_PATH, "r", encoding="utf-8") as f:
    script_text = f.read()

old_line = "GENERATE_FIGURES = False"
new_line = "GENERATE_FIGURES = True"

if old_line not in script_text:
    raise ValueError(
        f"Could not find exact line: {old_line}\n"
        "Please check the script and update the replacement text."
    )

patched_text = script_text.replace(old_line, new_line, 1)

with open(TEMP_SCRIPT_PATH, "w", encoding="utf-8") as f:
    f.write(patched_text)

print(f"\nCreated temporary run script: {TEMP_SCRIPT_PATH}")
print("Original GitHub script was not modified.")

# ====================================
# 13. RUN NESTED CV PIPELINE
# ====================================
print("\nRunning nested CV pipeline...\n")
!python "$TEMP_SCRIPT_PATH"

# ====================================
# 14. LIST GENERATED OUTPUTS
# ====================================
print("\nGenerated output directories/files:")
!ls -lh "$OUTPUT_DIR"

print("\nRecursive output structure:")
!find "$OUTPUT_DIR" -maxdepth 2 -type f | sort

# ====================================
# 15. ENVIRONMENT INFO
# ====================================
import numpy as np
import pandas as pd
import matplotlib
import seaborn as sns
import sklearn
import xgboost
import shap
import optuna
import requests
import pyarrow
import openpyxl

print("\nColab environment:")
print("Python:", sys.version)
print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("matplotlib:", matplotlib.__version__)
print("seaborn:", sns.__version__)
print("scikit-learn:", sklearn.__version__)
print("xgboost:", xgboost.__version__)
print("shap:", shap.__version__)
print("optuna:", optuna.__version__)
print("requests:", requests.__version__)
print("pyarrow:", pyarrow.__version__)
print("openpyxl:", openpyxl.__version__)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Repo already exists. Pulling latest changes...
/content/pediatric-lupus-flare-prediction-gene-expression-ml
Already up to date.

Repo contents:
docs  LICENSE  notebooks  outputs  README.md  requirements.txt	scripts

Scripts folder contents:
build_lupus_dataset.py		       lupus_nested_cv_pipeline.py
lupus_nested_cv_pipeline_colab_run.py

Deleted 1 old PNG file(s) from: /content/drive/MyDrive/lupus_ml_outputs/figures
/content/pediatric-lupus-flare-prediction-gene-expression-ml

Configuration:
REPO_PATH: /content/pediatric-lupus-flare-prediction-gene-expression-ml
SCRIPT_PATH: /content/pediatric-lupus-flare-prediction-gene-expression-ml/scripts/lupus_nested_cv_pipeline.py
TEMP_SCRIPT_PATH: /content/pediatric-lupus-flare-prediction-gene-expression-ml/scripts/lupus_nested_cv_pipeline_colab_run.py
DATA_PATH: /content/drive/MyDrive/lupus_project_outputs/lupus_final_